# Understading images and Controlling Output

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

PROVIDERS = {
    "groq": ("https://api.groq.com/openai/v1", "groq_api", "openai/gpt-oss-120b"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini_api_key", "gemini-3.8-flash")
    
    }


In [7]:

# 1. Imports and configuration
import os, time


# 2. Define text prompt and image

TEXT_PROMPT = """
A red apple is placed on a wooden table.
The image contains one apple.
"""

IMAGE_PATH = "D:/LLM/LLM_module_01/day2/data/test_image.jpg"

QUESTION = "Describe the main object and its color in one sentence."

# 3. Configure API clients

clients = {}

for provider, (base_url, api_key_name, model) in PROVIDERS.items():
    clients[provider] = OpenAI(
        api_key=os.getenv(api_key_name),
        base_url=base_url if base_url else None
    )

# 4. Run text experiments
for provider, client in clients.items():
    model = PROVIDERS[provider][2]

    start_time = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": TEXT_PROMPT}
        ]
    )

    end_time = time.perf_counter()

    print(f"\nProvider: {provider}")
    print("Response:", response.choices[0].message.content)
    print("Input tokens:", response.usage.prompt_tokens)
    print("Output tokens:", response.usage.completion_tokens)
    print("Time taken:", round(end_time - start_time, 3), "seconds")



Provider: groq
Response: **Alt‑text / Detailed description**

A single, ripe red apple rests on the surface of a wooden table. The apple is glossy and slightly reflective, showing a smooth, round shape with a short green stem protruding from its top. The wooden tabletop has a warm, honey‑colored grain pattern with faint, uneven lines that give it a natural, textured appearance. The lighting is soft and even, casting a subtle shadow of the apple onto the table, which helps to emphasize the fruit’s three‑dimensional form. No other objects are visible in the scene.
Input tokens: 88
Output tokens: 292
Time taken: 0.998 seconds

Provider: gemini
Response: That sounds like a clean, classic composition. 

How can I help you with this? I can:
- **Expand it into a detailed prompt** for an AI image generator (including lighting, style, camera angle, and textures).
- **Write a descriptive passage or story** based on this setting.
- **Answer questions** or help modify the scene.
Input tokens: 20


In [5]:
#decoding image
import base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")

In [ ]:
# 5. Run image experiments
encoded_image = encode_image(IMAGE_PATH)
PROVIDERS = {
    "groq": ("https://api.groq.com/openai/v1", "groq_api", "qwen/qwen3.8-27b"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini_api_key", "gemini-3.8-flash")}

for provider, client in clients.items():
    model = PROVIDERS[provider][2]
    print(f"Testing provider: {provider}, model: {model}")

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": QUESTION},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{encoded_image}"
                        }
                    }
                ]
            }
        ]
    )

    print(f"\nProvider: {provider}")
    print("Response:", response.choices[0].message.content)
    print("Input tokens:", response.usage.prompt_tokens)
    print("Output tokens:", response.usage.completion_tokens)




Testing provider: groq, model: qwen/qwen3.8-27b

Provider: groq
Response: The main object in the image is a shiny red apple.
Input tokens: 1817
Output tokens: 13
Testing provider: gemini, model: gemini-3.8-flash

Provider: gemini
Response: The main object is a bright red apple resting on the edge of a wooden table.
Input tokens: 1092
Output tokens: 17


In [11]:
import pandas as pd

df = pd.DataFrame([
    {
        "Model": "Groq - Qwen",
        "Input Type": "Text",
        "Input Tokens": 88,
        "Output Tokens": 292
    },
    {
        "Model": "Groq - Qwen",
        "Input Type": "Image",
        "Input Tokens": 1817,
        "Output Tokens": 13
    },
    {
        "Model": "Gemini",
        "Input Type": "Text",
        "Input Tokens": 20,
        "Output Tokens": 76
    },
    {
        "Model": "Gemini",
        "Input Type": "Image",
        "Input Tokens": 1092,
        "Output Tokens": 17
    }
])

print(df)

         Model Input Type  Input Tokens  Output Tokens
0  Groq - Qwen       Text            88            292
1  Groq - Qwen      Image          1817             13
2       Gemini       Text            20             76
3       Gemini      Image          1092             17


# Output control using Tempreature 

In [12]:
TEMPERATURE_PROMPT = """
Explain what a Python list is in simple words.
Give one example.
"""

TEMPERATURES = [0.0, 0.5, 1.0]

In [14]:
for provider, client in clients.items():
    model = PROVIDERS[provider][2]

    for temp in TEMPERATURES:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": TEMPERATURE_PROMPT}
            ],
            temperature=temp , 
            max_tokens=200
        )

        print(f"\nProvider: {provider} | Temperature: {temp}")
        print(response.choices[0].message.content)


Provider: groq | Temperature: 0.0
A Python list is like a **labeled box** that can hold multiple items in a specific order. You can add, remove, or change the items inside it, and you can access any item by its position (starting from 0).

### Example:
```python
fruits = ["apple", "banana", "cherry"]
print(fruits[0])  # Output: apple
```

In this example:
- `fruits` is the name of the list.
- It contains three items: `"apple"`, `"banana"`, and `"cherry"`.
- `fruits[0]` gets the first item, which is `"apple"`.

Provider: groq | Temperature: 0.5
A Python list is like a **numbered box** that can hold multiple items in a specific order. You can add, remove, or change items inside it, and you can access any item by its position (starting at index 0).

### Example:
```python
fruits = ["apple", "banana", "cherry"]
print(fruits[1])  # Output: banana
```

In this example:
- `fruits` is a list containing three strings.
- `fruits[1]` accesses the second item ("banana") because counting starts at